# Automated SST-1 Reduction code
## Camille Bishop
### Updated 12/3/22
This code will allow you to visualize a run, load the data, mask and integrate it (mask drawn in a separate notebook), and then quickly reduce to parameters that I believe are the most important to quickly look at a sample. The data loading is not streamlined in this notebook - Bijal has written a very nice loader that you may choose to combine. The main contribution of this notebook is everything written below the "Total Scan Output" line. Once all of those cells have been run, one can move to the bottom cell (underneath section "Putting everything together") and, __with a group of 4 scans, have a full analysis done in <5 minutes.__

### Important notes:
I think right now you can pip install both PyHyperScattering and xarray, but make sure this works for you before you rely on this.
In order to use these, you *must* have two scans in orthogonal polarization. If you don't, we can make some edits so that you can look at the scans separately, but they're not in here yet, and you really should have the orthogonal polarization if you want to learn from your data.
#### There are some weird variable specifications that you need to be careful about
Because of some last-minute doctoring, there are certain functions that are currently pretty dangerous. I've done my best to specify in the docs. These include situations such as specifying the lower polarization you're using, stating whether you want SAXS/WAXS combined, etc. If you have any questions, please email camille.e.bishop@gmail.com for beamline emergencies.

In [ ]:
import pickle

import sys

import numpy as np
import pandas as pd
import datetime

import matplotlib.pyplot as plt
import matplotlib as mpl

from matplotlib.colors import LogNorm
from matplotlib.pyplot import cm
from scipy.signal import savgol_filter


In [ ]:
!pip install PyHyperScattering
import PyHyperScattering
!pip install xarray==2022.3
import xarray as xr
load = PyHyperScattering.load.SST1RSoXSDB(corr_mode='none')

# Look at Run, choose scans
## The new code that Bijal just shared will be better for this

In [ ]:
print(f'Using PyHyperScattering Version: {PyHyperScattering.__version__}')

In [ ]:
# no databroker structure or anything here; modify notebook to just start from pickles
runlist = load.summarize_run(institution='CUBLDER',sample='Blend*')
#runlist = load.summarize_run(institution='NIST',sample='CB_4p1')
runlist

In [ ]:
runlist.loc[runlist['sample_id'].isin(['Blend9', 'Blend10', 'Blend11', 'Blend12'])].sort_values(by='scan_id', ascending=True)

In [ ]:
WAXSstack_P0 = load.loadSeries([45921],'sample_name')
WAXSstack_P90 = load.loadSeries([45922],'sample_name')
SAXSstack_P0 = load.loadSeries([45928],'sample_name')
SAXSstack_P90 = load.loadSeries([45929],'sample_name')

Mask was drawn in another notebook; ui was a little too clunky for my taste to include in this notebook

In [ ]:
SAXSpath = './October2022_Masks/CB_masks/CB_4p1_140_pol45SAXSmask.json'
WAXSpath = './October2022_Masks/CB_masks/CB_4p1_140_pol45WAXSmask.json'

In [ ]:
imgWAXS_P0 = WAXSstack_P0.unstack('system').sel(energy=284.,method='nearest').sel(sample_name='CB_LRZ_3')##.plot(norm=LogNorm(10,1000),ax=ax1)
imgWAXS_P90 = WAXSstack_P90.unstack('system').sel(energy=284.,method='nearest').sel(sample_name='CB_LRZ_3')
imgSAXS_P0 = SAXSstack_P0.unstack('system').sel(energy=284.,method='nearest').sel(sample_name='CB_LRZ_3')
imgSAXS_P90 = SAXSstack_P90.unstack('system').sel(energy=284.,method='nearest').sel(sample_name='CB_LRZ_3')#.plot(norm=LogNorm(10,1000),ax=ax2)

In [ ]:
WAXSinteg = PyHyperScattering.integrate.PFEnergySeriesIntegrator(geomethod='template_xr',template_xr = WAXSstack_P0) # this integrator works for both WAXS ones
SAXSinteg = PyHyperScattering.integrate.PFEnergySeriesIntegrator(geomethod='template_xr',template_xr = SAXSstack_P0) # this integrator works for both WAXS ones

In [ ]:
WAXSinteg.ni_beamcenter_x = 397.84 # Oct 2022 2nd run, visually determined beam center
WAXSinteg.ni_beamcenter_y = 550.18

SAXSinteg.ni_beamcenter_x = 497 # values for October 2022 SST1 run, as determined visually
SAXSinteg.ni_beamcenter_y = 510

# WAXSinteg.ni_beamcenter_x = 390.84 # values for October 2022 SST1 run
# WAXSinteg.ni_beamcenter_y = 555.18

# SAXSinteg.ni_beamcenter_x = 497 # values for October 2022 SST1 run, as determined visually
# SAXSinteg.ni_beamcenter_y = 520

In [ ]:
drawWAXS = PyHyperScattering.IntegrationUtils.DrawMask(imgWAXS_P0)
drawSAXS = PyHyperScattering.IntegrationUtils.DrawMask(imgSAXS_P0)
# drawWAXS = PyHyperScattering.IntegrationUtils.DrawMask(imgWAXS_P45)
# drawSAXS = PyHyperScattering.IntegrationUtils.DrawMask(imgSAXS_P45)

In [ ]:
%%capture
drawWAXS.load(WAXSpath)
drawSAXS.load(SAXSpath)


In [ ]:
WAXSinteg.mask = drawWAXS.mask
SAXSinteg.mask = drawSAXS.mask

In [ ]:
# integ.ni_beamcenter_x = 390.84
# integ.ni_beamcenter_y = 555.18


#fig, ax = plt.subplots(figsize=(12,10))

#PyHyperScattering.IntegrationUtils.Check.checkAll(WAXSinteg,imgWAXS_P45,img_min=10,img_max=100)
PyHyperScattering.IntegrationUtils.Check.checkAll(WAXSinteg,imgWAXS_P0,img_min=10,img_max=100)

In [ ]:
#PyHyperScattering.IntegrationUtils.Check.checkAll(SAXSinteg,imgSAXS_P45,img_min=10,img_max=100)
PyHyperScattering.IntegrationUtils.Check.checkAll(SAXSinteg,imgSAXS_P0,img_min=10,img_max=100)

In [ ]:
stack_intWAXS_P0 = WAXSinteg.integrateImageStack(WAXSstack_P0)
stack_intWAXS_P90 = WAXSinteg.integrateImageStack(WAXSstack_P90)
stack_intSAXS_P0 = SAXSinteg.integrateImageStack(SAXSstack_P0)
stack_intSAXS_P90 = SAXSinteg.integrateImageStack(SAXSstack_P90)

### At this point you can put the data into a pickle to avoid the long integration steps, and just re-open it later. Peter Beaucage warns against this, since they can break when code gets updated.. but I do it

In [ ]:
WAXS_p0 = stack_intWAXS_P0.unstack('system')
WAXS_p90 = stack_intWAXS_P90.unstack('system')
SAXS_p0 = stack_intSAXS_P0.unstack('system')
SAXS_p90 = stack_intSAXS_P90.unstack('system')

# Manually looking at energies to pick
### Helpful to go back and do this after you've done some preliminary reduction to find energies of interest

In [ ]:
energies = np.asarray(SAXS_p90.energy)
# this time it's a long carbon scan

interesting_indices = [0, 54, 75, 82, 88, 101,102,107,114,119,126] # FOR FULL CARBON
#interesting_indices = [0, 18, 26, 29, 32, 41, 42, 47, 49, 50, 51, 54] # FOR SHORT CARBON (55)
interesting_E = energies[interesting_indices]

# Total Scan Output
-I vs q all-space (every energy) 
-I vs q para and perp (every energy)
-A vs q ratio (every energy)  
-A vs E ratio (3 or 4 q-selections)  
-Maybe something nice would be, after looking at these, a choice of 5-10 2D scans to output  
-Graphical 2D plots of I vs chi & I vs q as a function of energy


# Printing 2D plots (summary and selected qxy's)

In [ ]:
# updated 12/3/22 to save or not
def save_2Dabstract(scan,displaymin=1,displaymax=5000,save=True,outputfolder='./PLA_Reduction_Results/dummy/'):   
    fig, [ax1, ax2] = plt.subplots(1,2,figsize=(15,5))
    scan.mean('q').plot(norm=LogNorm(displaymin,displaymax),ax=ax1)
    scan.mean('chi').plot(norm=LogNorm(displaymin,displaymax),ax=ax2)
    labelname = str(scan.attrs['sample_name'])+str(scan.attrs['rsoxs_config'])
    plt.suptitle(f'{labelname}, Polarization = {scan.polarization[0]}')
    if save==True:
        plt.savefig(f'{outputfolder}{labelname}_P{int(scan.polarization[0])}.png',dpi=100)

#def save_qxys(selected_E,outputfolder):
# go back to Bijal's code to pull these out

In [ ]:
#big_saveloc = '/nsls2/users/cbishop/PLA_Reduction_Results/' # DO NOT USE MY FOLDER
big_saveloc = pickle_loc
save_2Dabstract(SAXS_p135,displaymax=10000,save=False)

# I vs q all-space
With options to output all curves, as well as a function to make a summary plot at selected energies

In [ ]:
def make_SAXSWAXS(WAXS,WAXS_plus90,SAXS,SAXS_plus90,e,sf=400,SAXSmax = 0.0085,WAXSmin = 0.0085):
    '''
    make_SAXSWAXS: turns a set of 4 scans (as integrated stacks) into a stitched, summed
    UPDATED 12/3/22
    I vs q over all space curve.
    Args:
       WAXS, WAXS_plus90,... Integrated, unstacked scan data. Function has been "updated" just so that variable names reflect using
       any two orthogonal polarizations.

       sf: the scale factor to multiply the SAXS by. Will often have to be manually adjusted.
       e: the energy at which you wish to make the stitched curve
       SAXSmax & WAXSmin: Cutoff values for merging the scans.

    Returns:
        q: over all SAXS/WAXS range
        I: over all SAXS/WAXS range
    '''
    WAXS1 = WAXS.mean('chi').sel(energy=e,method='nearest')
    qWAXS = np.asarray(WAXS1['q']); IWAXS1 = np.asarray(WAXS1)
    WAXS2 = WAXS_plus90.mean('chi').sel(energy=e,method='nearest'); IWAXS2 = np.asarray(WAXS2)
    IWAXStot = IWAXS1 + IWAXS2

    SAXS1 = SAXS.mean('chi').sel(energy=e,method='nearest')
    qSAXS = np.asarray(SAXS1['q']); ISAXS1 = np.asarray(SAXS1)
    SAXS2 = SAXS_plus90.mean('chi').sel(energy=e,method='nearest'); ISAXS2 = np.asarray(SAXS2)
    ISAXStot = ISAXS1 + ISAXS2
    
    ms = np.where(qSAXS<SAXSmax); mw = np.where(qWAXS>WAXSmin)
    qSAXS = qSAXS[ms]; ISAXStot = ISAXStot[ms]*sf
    qWAXS = qWAXS[mw]; IWAXStot = IWAXStot[mw]
    q = np.concatenate((qSAXS,qWAXS)); I = np.concatenate((ISAXStot,IWAXStot))
    I = np.squeeze(I)
    
    return q,I

def plotsave_SAXSorWAXS(SWAXS, SWAXS_plus90,selected_E,saveTXT_folder='./PLA_Reduction_Results/',WAXS=True):
    """
    plotsave_SAXSWAXS: allows you to plot the results of make_SAXSWAXS at selected energies to
    make sure stitching parameters have been set correctly.
    Args:
        WAXS, WAXS_plus90...: 4 scans with SAXS & WAXS in both polarizations
        selected_E: Range of energies to test over
        save_plot: Boolean, option to save plot as a png
        save_text: Boolean, option to save I vs q curves for each energy
        savePNG_ and saveTXT_folder: Locations to save results; defaults to a default directory
        (Eventually I should write a functionality to check for a directory and make it, so that I can write more easily)
        special_note: Default blank, any string you'd like to add to the title of the plot
    """
    
    plt.close(); 
    fig, ax = plt.subplots(figsize=(10,8))
    plot_color = iter(cm.rainbow(np.linspace(0, 1,np.size(selected_E))))
    labelname = str(SWAXS.attrs['sample_name'])
    for e in selected_E:
        SWAXS1 = SWAXS.mean('chi').sel(energy=e,method='nearest')
        qSWAXS = np.asarray(SWAXS1['q']); ISWAXS1 = np.asarray(SWAXS1)
        SWAXS2 = SWAXS_plus90.mean('chi').sel(energy=e,method='nearest'); ISWAXS2 = np.asarray(SWAXS2)
        ISWAXStot = ISWAXS1 + ISWAXS2
        ax.plot(qSWAXS,ISWAXStot,'o',markersize=3,color=next(plot_color),label=f'{round(e,2)} eV')
        if save_text==True:
            to_save = np.column_stack((q,I))
            if WAXS==True:
                np.savetxt(f'{saveTXT_folder}{labelname}_{round(e,2)}eV_WAXSIvsq.txt',to_save)
            else:
                np.savetxt(f'{saveTXT_folder}{labelname}_{round(e,2)}eV_SAXSIvsq.txt',to_save)
    ax.set_title(f'{labelname}-{special_note}')
    ax.set_xlabel('q (A^-1)')
    ax.legend()
    ax.loglog()
    if save_plot==True:
        if WAXS==True:
            plt.savefig(f'{savePNG_folder}{labelname}_Ivsq_curvesWAXS.png')
        else:
            plt.savefig(f'{savePNG_folder}{labelname}_Ivsq_curvesSAXS.png')
    plt.show()
    

def plotsave_SAXSWAXS(WAXS,WAXS_plus90,SAXS,SAXS_plus90,selected_E,save_plot=False, save_text=False,
                  savePNG_folder='C:/Users/ceb10/Documents/Research/PLA_Project/Data_Reduction/Default_PNGs/',
                  saveTXT_folder='C:/Users/ceb10/Documents/Research/PLA_Project/Data_Reduction/Default_text/',
                  sf=400,SAXSmax=0.0085,WAXSmin = 0.0085,special_note=''):
    """
    plotsave_SAXSorWAXS: same as above, just if you have only SAXS or WAXS
    """
    plt.close(); 
    fig, ax = plt.subplots(figsize=(10,8))
    plot_color = iter(cm.rainbow(np.linspace(0, 1,np.size(selected_E))))
    labelname = str(WAXS.attrs['sample_name'])
    for e in selected_E:
        q, I = make_SAXSWAXS(WAXS,WAXS_plus90,SAXS,SAXS_plus90,e,sf=sf,SAXSmax=SAXSmax,WAXSmin=WAXSmin)
        ax.plot(q,I,'o',markersize=3,color=next(plot_color),label=f'{round(e,2)} eV')
        if save_text==True:
            to_save = np.column_stack((q,I))
            np.savetxt(f'{saveTXT_folder}{labelname}_{round(e,2)}eV_Ivsq.txt',to_save)
    print(np.shape(q),np.shape(I))
    ax.set_title(f'{labelname}-{special_note}')
    ax.set_xlabel('q (A^-1)')
    ax.legend()
    ax.loglog()
    if save_plot==True:
        plt.savefig(f'{savePNG_folder}{labelname}_Ivsq_curves.png')
    plt.show()
    # saves with all NaNs included       

In [ ]:
# plotsave_SAXSWAXS(WAXS_integrated_P45,WAXS_integrated_P135,SAXS_integrated_P45,SAXS_integrated_P135,interesting_E,savePNG_folder=f'{big_saveloc}SAXSWAXS_isotropic/',
#                   saveTXT_folder=f'{big_saveloc}SAXSWAXS_isotropic/',sf=90,save_plot=False,save_text=False)

plotsave_SAXSWAXS(WAXS_p45,WAXS_p135,SAXS_p45,SAXS_p135,interesting_E,savePNG_folder=f'{big_saveloc}SAXSWAXS_isotropic/',
                  saveTXT_folder=f'{big_saveloc}SAXSWAXS_isotropic/',sf=400,save_plot=False,save_text=False)

# Adapting for 0 and 90 here on 11/3/22

# I vs q Para and Perp

In [ ]:
'''
get_everything: helper function to find I vs. q in a specific direction
This is quadrant-specific - i.e., if the integration angle is 45, it looks at I vs q in the upper right quadrant of the detector. -135 is the lower left quadrant.

Args:
    integration_angle: detector angle to integrate over (center of a 90-degree wedge). For a single scan, this will be called 4 times over the
    course of the analysis. E.g., if you're analyzing a P=0 scan, this will be called for 0 and 180 to analyze I_paras, and 90 and -90 for I_perps.
    integrated_stack: an unstacked, integrated scan (either SAXS or WAXS, one polarization)
    energy: run at single energy
Return:
    q, I: I vs. q at the single direction of integration

--------------------------
paraperp_1pol_general: finds scattering intensity parallel and perpendicular to the polarization vector for a single scan.
Will still need to be combined with the partner scan to use for analysis to account for biaxiality.
Note that the polarizations that can be specified at SST1 are from 0 to 180, and the scattering data is reported from -180 to 180 to account for all quadrants.
Args:
    integrated_stack: an unstacked, integrated scan (either SAXS or WAXS, one polarization)
    stack_polarization: An integer quantity between 0 and 180 (the only allowable specifications at SST1)
    energy: this is run at a single energy
    chi_width: +/- degrees of the slice. This should always be 45 unless there is a compelling
    reason otherwise.
Return:
    q, Ipara & Iperp: The total scattering intensity parallel to and perpendicular to the polarization vector
    OF A SINGLE SCAN. DOES NOT ACCOUNT FOR BIAXIALITY.
----------------------
    
paraperp_2pol: finds scattering intensity parallel and perpendicular to the polarization vector for a pair of
orthogonal scans, on EITHER SAXS or WAXS
Args:
    integrated_stack, integrated_stack_plus90: scan of the same sample in orthogonal polarizations
    lower_pol: integer value of the lowest polarization scan (e.g., 0 if you have P = 0/90, 45 if you have P = 45/135). This could probably be automated. There is nothing
    in this function that checks that these are self-consistent. When it is called in savetxt_paraperpSAXSWAXS later, it does have a print message to remind the user there.
    energy: this is run at a single energy
Return:
    q, Ipara & Iperp: The total scattering intensity parallel to and perpendicular to the polarization vector
    OF A COMBINATION OF SCANS; should account for any sample biaxiality.
----------------------

paraperp_2pol_SAXSWAXS: uses the above helper functions to make Ipara and Iperp for entire SAXSWAXS curves
Args:
    WAXS, WAXS_plus90... Integrated, unstacked scan data
    e: the energy at which you wish to make the stitched curve
    SAXSmax & WAXSmin: Cutoff values for merging the scans.
    sf: the scale factor to multiply the SAXS by. Will often have to be manually adjusted.
    lower_pol: integer value of the lowest polarization scan (e.g., 0 if you have P = 0/90, 45 if you have P = 45/135). This could probably be automated. There is nothing
    in this function that checks that these are self-consistent. When it is called in savetxt_paraperpSAXSWAXS later, it does have a print message to remind the user there.
    combined: True if you have both SAXS and WAXS, False if you have SAXS or WAXS only
    WAXS_bool: True if you have WAXS only, False if you have SAXS only (does not get used if combined=True)
 
Returns:
    q: over all SAXS/WAXS range
    I: over all SAXS/WAXS range
    
    For chi ranges from -180 to 180, so one difficulty is needing to take a wedge from 135 to 180 + -180 to -135 for a full 90 degrees
    '''

def get_everything(integration_angle,integrated_stack,energy):
    q = []; I = []
    if np.logical_and(integration_angle >= -135, integration_angle <= 135) == True:
        Iq = integrated_stack.sel(energy=energy,chi=slice(integration_angle - 45,integration_angle + 45)).mean(dim='chi')
        q = Iq['q']; I = np.asarray(Iq)
    elif integration_angle == 180 or integration_angle == -180:
        Iq_1 = integrated_stack.sel(energy=energy,chi=slice(135,180)).mean(dim='chi')
        Iq_2 = integrated_stack.sel(energy=energy,chi=slice(-180,-135)).mean(dim='chi')
        q = Iq_1['q']; I = np.asarray(Iq_1) + np.asarray(Iq_2)
    elif integration_angle > 135:
        Iq_1 = integrated_stack.sel(energy=energy,chi=slice(integration_angle,180)).mean(dim='chi')
        Iq_2 = integrated_stack.sel(energy=energy,chi=slice(-180,integration_angle-270)).mean(dim='chi')
        #q = Iq_1['q']; I = np.asarray(Iq_1) + np.asarray(Iq_2)
        q = Iq_1['q']; I = np.asarray(Iq_1) + np.asarray(Iq_2)
    elif pol_direction < -135:
        Iq_1 = integrated_stack.sel(energy=energy,chi=slice(-180,integration_angle)).mean(dim='chi')
        Iq_2 = integrated_stack.sel(energy=energy,chi=slice(180 - np.abs(-180-integration_angle),180)).mean(dim='chi')
        q = Iq_1['q']; I = np.asarray(Iq_1) + np.asarray(Iq_2)
    return q, I

def paraperp_1pol_general(integrated_stack,stack_polarization,energy,chi_width = 45):
    if stack_polarization > 180 or stack_polarization < 0:
        print('Specified polarization outside of allowable SST1 range')
    para_dir1 = stack_polarization; para_dir2 = stack_polarization - 180
    perp_dir1 = stack_polarization + 90; perp_dir2 = stack_polarization - 90
    if para_dir2 > 180:
        para_dir2 = para_dir2 - 360
    if perp_dir1 > 180:
        perp_dir1 = perp_dir1 - 360
    if perp_dir2 > 180:
        perp_dir2 = perp_dir2 - 360
    # since
    q, I_para_dir1 = get_everything(para_dir1,integrated_stack,energy); q, I_para_dir2 = get_everything(para_dir2,integrated_stack,energy)
    paratot = I_para_dir1 + I_para_dir2
    q, I_perp_dir1 = get_everything(perp_dir1,integrated_stack,energy); q, I_perp_dir2 = get_everything(perp_dir2,integrated_stack,energy)
    perptot = I_perp_dir1 + I_perp_dir2
    return q, paratot,perptot

                                                            
def paraperp_2pol(integrated_stack,integrated_stack_plus90,energy,lower_pol):
    q, para1, perp1 = paraperp_1pol_general(integrated_stack,stack_polarization=lower_pol,energy=energy)
    q, para2, perp2 = paraperp_1pol_general(integrated_stack_plus90,stack_polarization=lower_pol + 90,energy=energy)
    para_truetot = para1+para2
    perp_truetot = perp1+perp2
    
    return q, para_truetot, perp_truetot

def paraperp_2pol_SAXSWAXS(WAXS,WAXS_plus90,SAXS,SAXS_plus90,energy,lower_pol=0,SAXSmax=0.0085,WAXSmin=0.0085,sf=400,combined=True,WAXS_bool=True):
    
    Ipara=[]; Iperp=[]; q=[]

    
    if combined==True:
        qWAXS, paraWAXS, perpWAXS = paraperp_2pol(WAXS,WAXS_plus90,energy,lower_pol)
        qSAXS, paraSAXS, perpSAXS = paraperp_2pol(SAXS,SAXS_plus90,energy,lower_pol)
        ms = np.where(qSAXS<SAXSmax); mw = np.where(qWAXS>WAXSmin)
        qSAXS = qSAXS[ms]; ISAXSpara = paraSAXS[ms]*sf; ISAXSperp = perpSAXS[ms]*sf
        qWAXS = qWAXS[mw]; IWAXSpara = paraWAXS[mw]; IWAXSperp = perpWAXS[mw]
        q = np.concatenate((qSAXS,qWAXS))
        Ipara = np.concatenate((ISAXSpara,IWAXSpara))
        Iperp = np.concatenate((ISAXSperp,IWAXSperp))
        Ipara = np.squeeze(Ipara); Iperp = np.squeeze(Iperp)
    elif combined==False and WAXS_bool==True:
        q, Ipara, Iperp = paraperp_2pol(WAXS,WAXS_plus90,energy,lower_pol)
    elif combined==False and WAXS_bool==False:
        q, Ipara, Iperp = paraperp_2pol(SAXS,SAXS_plus90,energy,lower_pol)
        
    # qSAXS, paraSAXS, perpSAXS = paraperp_2pol(SAXS,SAXS_plus90,energy,lower_pol)
    
    return q, Ipara, Iperp
        
# -I vs q para and perp (every energy)    

In [ ]:
'''
savetxt_paraperpSAXSWAXS: allows you to output Ipara and Iperp at selected energies to text files.
No informative plotting functionality; however, it does plot one curve (arbitrarily chosen to be the first energy
above 299 eV) so that you can check that the scaling makes sense.
Args:
    WAXSint45...: 4 scans with SAXS & WAXS in both polarizations
    selected_E: Range of energies to test over
    saveTXT_folder: Location to save results; defaults to a default directory
    (Eventually I should write a functionality to check for a directory and make it, so that I can write more easily)
    SAXSmax & WAXSmin: Cutoff values for merging the scans.
    sf: the scale factor to multiply the SAXS by. Will often have to be manually adjusted.
    lower_pol: integer value of the lowest polarization scan (e.g., 0 if you have P = 0/90, 45 if you have P = 45/135). This could probably be automated. There is a print
    statement to remind the user, but it will not stop executing if it's incorrect.
    
Returns:
    No returns, but saves text files. Outputs a plot in-notebook to visually check scaling.

'''

def savetxt_paraperpSAXSWAXS(WAXS,WAXS_plus90,SAXS,SAXS_plus90,selected_E,lower_pol,savetext=False,
                  saveTXT_folder='C:/Users/ceb10/Documents/Research/PLA_Project/Data_Reduction/Default_text/',
                  sf=400,SAXSmax=0.0085,WAXSmin = 0.0085,combined=True,WAXS_bool=True):
    labelname = str(WAXS.attrs['sample_name'])
    print(f'The lower polarization you have will give correct results only for P = {lower_pol},{lower_pol+90} scan pairs')
    plt.close()
    i = 0
    for e in selected_E:
        q, Ipara, Iperp = paraperp_2pol_SAXSWAXS(WAXS,WAXS_plus90,SAXS,SAXS_plus90,e, lower_pol=lower_pol,
                                                 SAXSmax=SAXSmax,WAXSmin=WAXSmin,sf=sf,combined=combined,WAXS_bool=WAXS_bool)
        to_save_para = np.column_stack((q,Ipara))
        to_save_perp = np.column_stack((q,Iperp))
        if savetext==True:
            np.savetxt(f'{saveTXT_folder}{labelname}_{round(e,2)}eV_Iparavsq.txt',to_save_para)
            np.savetxt(f'{saveTXT_folder}{labelname}_{round(e,2)}eV_Iperpvsq.txt',to_save_perp)
        if (e > 280) and i < 1: # plotting a single set of curves to make sure that things are scaled right
            plt.plot(q,Ipara,'k','o',markersize=3)
            plt.plot(q,Iperp,'red','o',markersize=3)
            i = 100
    plt.loglog()
    plt.show()
    # saves with all NaNs included 

In [ ]:
savetxt_paraperpSAXSWAXS(WAXS_p0,WAXS_p90,SAXS_p0,SAXS_p90,interesting_E,lower_pol=0,sf=200,saveTXT_folder=f'{big_saveloc}Garbagerun/',combined=True,
                        WAXS_bool=True)

# A vs q
-A vs q ratio (every energy)  

In [ ]:
'''
THESE CALCULATIONS ARE NOT UP TO QUANTITATIVE STANDARDS YET

oldfashioned_AR: calculates A vs q in the old fashioned way (still using +/- 45 degree wedges)

Args:
    Arguments are essentially all the same as above.
    * single energy
    * could add savitsky-golay smoothing window sizes and orders in the future
    
Returns:
    q, Anisotropy ratio

'''

def oldfashioned_AR(WAXS,WAXS_plus90,SAXS,SAXS_plus90,energy,lower_pol=0,SAXSmax=0.0085,WAXSmin=0.0085,sf=400):
    q, Ipara, Iperp = paraperp_2pol_SAXSWAXS(WAXS,WAXS_plus90,SAXS,SAXS_plus90,energy,lower_pol=lower_pol,SAXSmax=SAXSmax,WAXSmin=WAXSmin,sf=sf)
    # squeezed already in prior functions
    m = np.where(~np.isnan(Iperp))
    q = q[m]; Ipara = Ipara[m]; Iperp=Iperp[m]
    Ipara_smooth = savgol_filter(Ipara,15,3)
    Iperp_smooth = savgol_filter(Iperp,15,3)
    aniso_smooth = (Ipara_smooth - Iperp_smooth) / (Ipara_smooth + Iperp_smooth)
    return q, aniso_smooth
# could mess with smoothing window

def plotsave_AR(WAXS,WAXS_plus90,SAXS,SAXS_plus90,selected_E,lower_pol=0,save_plot=False, save_text=False,
                  savePNG_folder='C:/Users/ceb10/Documents/Research/PLA_Project/Data_Reduction/Default_PNGs/',
                  saveTXT_folder='C:/Users/ceb10/Documents/Research/PLA_Project/Data_Reduction/Default_text/',
                  sf=400,SAXSmax=0.0085,WAXSmin = 0.0085,special_note=''):
    plt.close(); 
    fig, ax = plt.subplots(figsize=(10,8))
    plot_color = iter(cm.rainbow(np.linspace(0, 1,np.size(selected_E))))
    labelname = str(WAXS.attrs['sample_name'])
    for e in selected_E:
        q, A = oldfashioned_AR(WAXS,WAXS_plus90,SAXS,SAXS_plus90,e,lower_pol=lower_pol,sf=sf,SAXSmax=SAXSmax,WAXSmin=WAXSmin)
        ax.plot(q,A,'o',markersize=3,color=next(plot_color),label=f'{round(e,2)} eV')
        if save_text==True:
            to_save = np.column_stack((q,A))
            np.savetxt(f'{saveTXT_folder}{labelname}_{round(e,2)}eV_Avsq.txt',to_save)
    ax.set_title(f'{labelname}-{special_note}')
    ax.set_xlabel('q (A^-1)')
    ax.legend()
    ax.semilogx()
    ax.set_ylim(-1,1)
    if save_plot==True:
        plt.savefig(f'{savePNG_folder}{labelname}_Avsq_curves.png')
    plt.show()
    

In [ ]:
plotsave_AR(WAXS_p45,WAXS_p135,SAXS_p45,SAXS_p135,interesting_E,lower_pol=45,
            sf=390,save_plot=False,save_text=False,savePNG_folder=f'{big_saveloc}SAXSWAXS_Avsq/',saveTXT_folder=f'{big_saveloc}SAXSWAXS_Avsq/')

# A vs E
Input q bounds, get out A vs E

In [ ]:
'''
Functions to turn the unstacked scans into I vs chi, normalized to avoid biaxiality. These all work in the
original inverse Angstroms from the experimental data; there are old codes that do nm conversions.

quick_break: returns I vs q for a wedge of specified width on a specified scan (us. Ipara, Iperp).
Args:
    unstack: an unstacked sample scan
    energy: energy for evaluation
    center: center angle of the chi wedge
    width: width of wedge
Returns:
    q, I: at specified angle and wedge width
    
find_norm_sum: used to find the total value of intensity over all scattering angles at the q-range of anisotropy calculation
Args:
    unstack: an unstacked sample scan
    energy: to evaluate at
    qbin_center: center of q bin; ensure that it's the same as the thing you're normalizing
    bin_width: +/- to above (qbin_center = 0.05, bin_width = 0.01 will integrate from 0.04 to 0.06)
Returns:
    normalizer: total sum of all intensity in the difference curve
    
make_norm_array: makes an array of all normalization factors at every energy. Same args as above.
Returns:
    normalizations: an array of normalization factors at every energy specified

get_azimuthal anisotropy: returns the difference curve between the two intensities vs. chi. Normalizes for sample biaxiality
Args:
    unstacked, unstacked_plus90: unstacked sample scan and its orthogonal buddy
    E: the energy to evaluate at
    qbounds: max and min of range (note that I switch between defining this way and qcenter/binwidth)
Returns:
    chi: scattering angle
    difference_I: the difference between the two intensities.
'''

def quick_break(unstack,energy,center,width=45):
    integration = unstack.rsoxs.slice_chi(center,width).sel(energy=energy,method='nearest')
    q= np.asarray(integration['q']); I = np.asarray(integration); I = np.squeeze(I)
    return q, I

def find_norm_sum(unstacked,energy,qbin_center,bin_width): # needs the bin_width is plus-minus; e.g., if qbin_center = 0.05 and bin_width = 0.01, will integrate 0.04 to 0.06 A^-1
    q, I = quick_break(unstacked,energy,center=180,width=180) # returns
    m = np.where(np.logical_and((q > qbin_center-bin_width),(q < qbin_center+bin_width)))
    normalizer = np.sum(I[m])
    return normalizer

def make_norm_array(unstacked,unstacked_plus90,energies,qbin_center,bin_width):
    normalizations = []
    for e in energies:
        norm = find_norm_sum(unstacked,e,qbin_center=qbin_center,bin_width=bin_width)
        norm_plus90 = find_norm_sum(unstacked_plus90,e,qbin_center=qbin_center,bin_width=bin_width)
        total_norm = norm + norm_plus90
        normalizations = np.append(normalizations,total_norm)
    return normalizations

# now, we want to take in the stacks and find the azimuthal anisotropy

def get_azimuthal_anisotropy(unstacked, unstacked_plus90,E,qbounds): # qbounds is a tuple; in A^-1 (experimental units)
    lowq = qbounds[0]; highq = qbounds[1]
    pol_curve = unstacked.sel(energy=E,method='nearest').sel(q=slice(lowq,highq)).mean('q')
    pol_curve_plus90 = unstacked_plus90.sel(energy=E,method='nearest').sel(q=slice(lowq,highq)).mean('q')    
    chi = np.asarray(pol_curve['chi']);
    I1 = np.asarray(pol_curve); I1_plus90 = np.asarray(pol_curve_plus90)
    difference_I = np.subtract(I1,I1_plus90); difference_I = np.squeeze(difference_I)
    
    return chi, difference_I # so this will be the difference I curve at a given energy

In [ ]:
'''
Functions to calculate the anisotropy at a given energy, and a range of energies.

calc_anisotropy_sum: finds anisotropy (with phase) at a given energy. Sums up the absolute value of I differences from 0.
Args:
    unstacked, unstacked_plus90: orthogonal scans
    E: energy
    qbounds: min and max (A^-1)
    lowerpol: value of the lowest polarization
    
Returns:
    aniso_sum: sum of all intensity differences, with phase
    
make_AvsE: makes a curve for all energies. Note that this takes some time to execute.

'''

# just did at 5:45 pm 12/3/2022
def calc_anisotropy_sum(unstacked, unstacked_plus90,E,qbounds,lowerpol=0): # sum up all the area of anisotropy, and assign a sign
    chi, diffI = get_azimuthal_anisotropy(unstacked, unstacked_plus90, E, qbounds)
    mpos = np.where(np.logical_and(~np.isnan(diffI),diffI > 0))
    mneg = np.where(np.logical_and(~np.isnan(diffI),diffI < 0))
    aniso_sum = np.abs(np.sum(diffI[mpos]))+np.abs(np.sum(diffI[mneg]))
    # check phase; maximum at chi = lowerpol angle is a positive anisotropy
    lowpol_index = np.where(np.logical_and(~np.isnan(diffI),(np.logical_and(chi > (lowerpol-10),chi < (lowerpol+10)))))
    phase_sum = np.sum(diffI[lowpol_index])
    if phase_sum < 0:
        aniso_sum = -1*aniso_sum
    return aniso_sum

def make_AvsE(unstacked,unstacked_plus90,energies,qbounds,lower_pol=0,normalize=True,savetext=False,savePNG=False):
    anisotropies = []
    labelname = str(unstacked.attrs['sample_name'])
    for e in energies:
        aniso_sum = calc_anisotropy_sum(unstacked,unstacked_plus90,e,qbounds,lowerpol=lower_pol)
        anisotropies = np.append(anisotropies,aniso_sum)
    if normalize==True:
        qbin_center = (qbounds[0]+qbounds[1])/2.
        bin_width= qbounds[1] - qbin_center
        normalizers = make_norm_array(unstacked,unstacked_plus90,energies,qbin_center,bin_width)
        anisotropies = anisotropies/normalizers
    if savetext==True:
        to_save = np.column_stack((energies,anisotropies))
        np.savetxt(f'{saveTXT_folder}{labelname}_q{qbounds[0]}to{qbounds[1]}_AvsE.txt',to_save)
    plt.close()
    fig, ax = plt.subplots(figsize=(5,4))
    ax.plot(energies,anisotropies,'o')
    ax.set_title(f'{labelname}, q = {qbounds}')
    ax.set_xlabel('Energy (eV)')
    if savePNG==True:
        plt.savefig(f'{savePNG_folder}{labelname}_q{qbounds[0]}to{qbounds[1]}_AvsE.png')
    plt.show()

# Sample usage of all
### By specifying an output folder, you can get PNGs and/or text summaries of everything by running the following cell.

I vs q

In [ ]:
plotsave_SAXSWAXS(WAXS_p0,WAXS_p90,SAXS_p0,SAXS_p90,interesting_E,savePNG_folder=f'{big_saveloc}SAXSWAXS_isotropic/',
                  saveTXT_folder=f'{big_saveloc}SAXSWAXS_isotropic/',sf=400,save_plot=False,save_text=False)
savetxt_paraperpSAXSWAXS(WAXS_p0,WAXS_p90,SAXS_p0,SAXS_p90,interesting_E,lower_pol=0,sf=200,saveTXT_folder=f'{big_saveloc}Garbagerun/',combined=True,
                        WAXS_bool=True)
# note that the above only displays 2 curves on the graph for clarity, but does save all out as text files
plotsave_AR(WAXS_p0,WAXS_p90,SAXS_p0,SAXS_p90,interesting_E,lower_pol=0,
            sf=390,save_plot=False,save_text=False,savePNG_folder=f'{big_saveloc}SAXSWAXS_Avsq/',saveTXT_folder=f'{big_saveloc}SAXSWAXS_Avsq/')
make_AvsE(WAXS_p0,WAXS_p90,energies,(0.015,0.03),lower_pol=0,normalize=True,savetext=False,savePNG=False)
